In [8]:
import pandas as pd

IBI_csv = pd.read_csv("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_lur_sentinel_features_250m_with_upper_left_corner.csv")
IVI_daily_parquet = pd.read_parquet("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_sentinel_features_daily_250m_filtered.parquet")
IVI_static_parquet = pd.read_parquet("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_sentinel_features_static_250m_filtered.parquet")
grid_basic = pd.read_csv("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/격자_250m_4326.csv")

In [2]:
IBI_csv.head()

,grid_id,grid_num,grid_row,grid_col,center_lat,center_lon,center_x_m,center_y_m,ul_lat,ul_lon,ul_x_m,ul_y_m,NDVI_mean,IBI_mean,NDBI_mean,MNDWI_mean
0,g_1,1,0,1,37.701397,126.764062,935125.0,1967125.0,37.702514,126.762632,935000.0,1967250.0,0.057780,0.617521,-0.084395,0.061064
1,g_2,2,0,2,37.701414,126.766897,935375.0,1967125.0,37.702532,126.765468,935250.0,1967250.0,0.111134,0.533825,-0.033536,-0.050084
2,g_3,3,0,3,37.701432,126.769733,935625.0,1967125.0,37.702550,126.768304,935500.0,1967250.0,0.179570,0.610423,-0.025159,-0.141717
3,g_4,4,0,4,37.701450,126.772569,935875.0,1967125.0,37.702567,126.771140,935750.0,1967250.0,0.375574,0.846456,-0.161136,-0.167444
4,g_5,5,0,5,37.701467,126.775405,936125.0,1967125.0,37.702585,126.773976,936000.0,1967250.0,0.481719,0.882575,-0.243069,-0.162886


TypeError: 'tuple' object is not callable

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import os

# ── 파일 로드 ──────────────────────────────────────────────────────────────
grid_basic = pd.read_csv(
    "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/격자_250m_4326.csv"
)
ibi_csv = pd.read_csv(
    "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_lur_sentinel_features_250m_with_upper_left_corner.csv"
)

# ── 최근접 좌표 매칭 (KD-Tree) ────────────────────────────────────────────
# grid_basic의 각 격자에 대해 ibi_csv의 가장 가까운 격자를 찾아 NDVI_mean, IBI_mean 붙이기
tree = cKDTree(ibi_csv[['center_lat', 'center_lon']].values)
dists, idxs = tree.query(grid_basic[['lat', 'lon']].values)

print(f"매칭 거리 평균: {(dists * 111):.3f} km" if dists.ndim == 0
      else f"매칭 거리 평균: {(dists * 111).mean():.3f} km  |  최대: {(dists * 111).max():.3f} km")

# ── 변수 붙이기 ───────────────────────────────────────────────────────────
grid_final = grid_basic.copy()
grid_final['NDVI'] = ibi_csv.iloc[idxs]['NDVI_mean'].values
grid_final['IBI']  = ibi_csv.iloc[idxs]['IBI_mean'].values

print(f"\n최종 데이터 shape: {grid_final.shape}")
print(grid_final[['fid', 'lat', 'lon', 'NDVI', 'IBI']].head())
print(f"\nNDVI  —  min: {grid_final['NDVI'].min():.4f}  max: {grid_final['NDVI'].max():.4f}  null: {grid_final['NDVI'].isna().sum()}")
print(f"IBI   —  min: {grid_final['IBI'].min():.4f}  max: {grid_final['IBI'].max():.4f}  null: {grid_final['IBI'].isna().sum()}")

# ── 저장 (grid_basic과 동일 위치) ─────────────────────────────────────────
save_path = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/격자_250m_4326_with_lur.csv"
grid_final.to_csv(save_path, index=False)
print(f"\n저장 완료 → {save_path}")

In [ ]:
import pandas as pd
import numpy as np
import pickle
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
import os

SAVE_DIR = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본"

# ── 1. ST-GNN 전체 타임스탬프 로드 ───────────────────────────────────────
with open('/workspace/ST-GNN Modeling/split_info.pkl', 'rb') as f:
    split = pickle.load(f)

all_times = sorted(set(split['train_times']) | set(split['valid_times']) | set(split['test_times']))
all_times_dt = pd.DatetimeIndex(all_times)
all_times_f  = np.array([pd.Timestamp(t).timestamp() for t in all_times], dtype=np.float64)
print(f"ST-GNN 타임스텝: {len(all_times)}개  ({all_times[0]} ~ {all_times[-1]})")

# ── 2. 위성 데이터 로드 & pivot ──────────────────────────────────────────
daily = pd.read_parquet(
    "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_sentinel_features_daily_250m_filtered.parquet"
)
daily['date'] = pd.to_datetime(daily['date'])

ndvi_wide = daily.pivot(index='date', columns='grid_id', values='NDVI').sort_index()  # [70, 18179]
ibi_wide  = daily.pivot(index='date', columns='grid_id', values='IBI').sort_index()

sat_times_f = np.array([t.timestamp() for t in ndvi_wide.index], dtype=np.float64)
print(f"위성 날짜: {len(sat_times_f)}개  ({ndvi_wide.index[0].date()} ~ {ndvi_wide.index[-1].date()})")

# ── 3. 선형 보간 (전체 격자 벡터화) ─────────────────────────────────────
# boundary: ST-GNN 기간이 위성 데이터 기간 안에 완전히 포함되므로 extrapolation 불필요
def interpolate_to_stgnn(wide_df, sat_times_f, all_times_f):
    values = wide_df.values.astype(np.float32)          # [70, G_ibi]
    interpolator = interp1d(
        sat_times_f, values,
        axis=0, kind='linear',
        bounds_error=False,
        fill_value=(values[0], values[-1]),              # 경계 밖은 첫/마지막 값 유지
    )
    return interpolator(all_times_f).astype(np.float32) # [T, G_ibi]

print("NDVI 보간 중...")
ndvi_interp = interpolate_to_stgnn(ndvi_wide, sat_times_f, all_times_f)  # [T, 18179]
print("IBI  보간 중...")
ibi_interp  = interpolate_to_stgnn(ibi_wide,  sat_times_f, all_times_f)  # [T, 18179]
print(f"보간 완료: {ndvi_interp.shape}  dtype={ndvi_interp.dtype}")

# ── 4. grid_basic 순서로 재정렬 (KD-Tree 매칭) ──────────────────────────
ibi_csv = pd.read_csv(
    "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/IBI, NVDI/grid_lur_sentinel_features_250m_with_upper_left_corner.csv"
)
grid_basic = pd.read_csv(os.path.join(SAVE_DIR, "격자_250m_4326.csv"))

tree = cKDTree(ibi_csv[['center_lat', 'center_lon']].values)
_, idxs = tree.query(grid_basic[['lat', 'lon']].values)          # grid_basic[i] → ibi_csv[idxs[i]]

# ibi_csv index → ndvi_wide column 위치
grid_ids_for_basic = ibi_csv.iloc[idxs]['grid_id'].values        # [G_basic] ibi grid_id 배열
col_order = list(ndvi_wide.columns)
col_positions = np.array([col_order.index(gid) for gid in grid_ids_for_basic])

ndvi_final = ndvi_interp[:, col_positions]   # [T, G_basic=10125]
ibi_final  = ibi_interp[:, col_positions]    # [T, G_basic=10125]
print(f"grid_basic 기준 재정렬: {ndvi_final.shape}")

# ── 5. 저장 ──────────────────────────────────────────────────────────────
np.save(os.path.join(SAVE_DIR, "ndvi_hourly.npy"),      ndvi_final)
np.save(os.path.join(SAVE_DIR, "ibi_hourly.npy"),       ibi_final)
np.save(os.path.join(SAVE_DIR, "timestamps_all.npy"),   np.array(all_times))

# split별 인덱스도 저장 (학습 시 분리 사용)
for split_name, times in [('train', split['train_times']),
                           ('val',   split['valid_times']),
                           ('test',  split['test_times'])]:
    times_sorted = sorted(times)
    idxs_split = np.array([all_times.index(t) for t in times_sorted])
    np.save(os.path.join(SAVE_DIR, f"time_idx_{split_name}.npy"), idxs_split)
    print(f"  {split_name}: {len(idxs_split)}개 타임스텝 인덱스 저장")

print(f"\n저장 완료 → {SAVE_DIR}/")
print(f"  ndvi_hourly.npy  {ndvi_final.shape}  {ndvi_final.nbytes/1e6:.0f} MB")
print(f"  ibi_hourly.npy   {ibi_final.shape}   {ibi_final.nbytes/1e6:.0f} MB")

# 보간 품질 확인
print(f"\n=== 보간 품질 확인 ===")
print(f"NDVI 범위: {ndvi_final.min():.4f} ~ {ndvi_final.max():.4f}")
print(f"IBI  범위: {ibi_final.min():.4f} ~ {ibi_final.max():.4f}")
print(f"NDVI NaN: {np.isnan(ndvi_final).sum()}")
print(f"IBI  NaN: {np.isnan(ibi_final).sum()}")